# Fine-tune DeBERTa-v3-base on ContractNLI (ClauseGuard verification module)

Run in **Google Colab** with a GPU runtime. This machine has Hugging Face access, unlike the deployment laptop (constraint 0.3). The fine-tuned model is saved as a self-contained local folder loaded with **no network access** on the deployment side.

**Data**: ContractNLI's own official `train.json` / `dev.json` / `test.json` (607 real contracts total, 17 fixed NDA hypotheses each, real `(premise, hypothesis, label)` entailment triples — see `docs/DATA_PROVENANCE.md`). Upload all three files from `data/raw/contractnli/contract-nli/` when prompted.

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate evaluate torch sentencepiece

In [ ]:
from google.colab import files
print("Upload train.json, dev.json, and test.json (from data/raw/contractnli/contract-nli/)")
uploaded = files.upload()

In [ ]:
import json

def load_triples(path, split_name):
    with open(path) as f:
        data = json.load(f)
    hypotheses = data["labels"]
    rows = []
    for doc in data["documents"]:
        spans = doc["spans"]
        for hyp_id, ann in doc["annotation_sets"][0]["annotations"].items():
            premise = " ".join(doc["text"][spans[i][0]:spans[i][1]] for i in ann.get("spans", []) if i < len(spans))
            rows.append({
                "premise": premise if premise.strip() else doc["text"][:2000],  # NotMentioned: no cited span, give document context
                "hypothesis": hypotheses[hyp_id]["hypothesis"],
                "label": ann["choice"],
                "split": split_name,
            })
    return rows

train_rows = load_triples("train.json", "train")
val_rows = load_triples("dev.json", "val")
test_rows = load_triples("test.json", "test")
print(f"train={len(train_rows)} val={len(val_rows)} test={len(test_rows)}")
print("train label distribution:", {l: sum(1 for r in train_rows if r['label'] == l) for l in ['Entailment','Contradiction','NotMentioned']})

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"
label2id = {"NotMentioned": 0, "Entailment": 1, "Contradiction": 2}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(rows):
    df = pd.DataFrame(rows)
    df["labels"] = df["label"].map(label2id)
    return Dataset.from_pandas(df[["premise", "hypothesis", "labels"]])

def tokenize(batch):
    return tokenizer(batch["premise"], batch["hypothesis"], truncation=True, max_length=384, padding="max_length")

train_ds = to_dataset(train_rows).map(tokenize, batched=True)
val_ds = to_dataset(val_rows).map(tokenize, batched=True)
test_ds = to_dataset(test_rows).map(tokenize, batched=True)

cols = ["input_ids", "attention_mask", "labels"]
for ds in (train_ds, val_ds, test_ds):
    ds.set_format(type="torch", columns=cols)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from scipy.special import softmax

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=-1)
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    try:
        auroc = roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    except ValueError:
        auroc = float("nan")  # can happen if a class is missing from a small eval batch
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1_macro": f1, "precision_macro": precision, "recall_macro": recall, "auroc_macro_ovr": auroc}

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

SEED = 42
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

args = TrainingArguments(
    output_dir="./deberta-contractnli-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    num_train_epochs=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    fp16=True,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_loss = [(h["epoch"], h["loss"]) for h in history if "loss" in h]
eval_loss = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]
eval_f1 = [(h["epoch"], h["eval_f1_macro"]) for h in history if "eval_f1_macro" in h]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(*zip(*train_loss), label="train_loss")
axes[0].plot(*zip(*eval_loss), label="eval_loss")
axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].set_title("Loss")
axes[1].plot(*zip(*eval_f1), label="eval_f1_macro", color="green")
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].set_title("Validation Macro-F1")
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

In [ ]:
test_metrics = trainer.evaluate(test_ds, metric_key_prefix="test")
print(test_metrics)

import json
with open("test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

## Ablation: verifier-off baseline reference

The brief requires an explicit ablation comparing hallucinated/false-obligation rate **with the verifier enabled vs. disabled**. This notebook trains the verifier; the ablation itself (running the extraction+agent pipeline with and without gating on this model's verdict) belongs in the backend pipeline, not here — but record the verifier's own held-out accuracy/F1 above as the reference point that ablation will be measured against.

## Save as a self-contained local folder (constraint 0.3)

In [ ]:
SAVE_DIR = "./clauseguard-verifier"
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import shutil
shutil.copy("training_curves.png", SAVE_DIR)
shutil.copy("test_metrics.json", SAVE_DIR)

shutil.make_archive("clauseguard-verifier", "zip", SAVE_DIR)
files.download("clauseguard-verifier.zip")

## Next steps (back on your own machine)

1. Unzip into `models/clauseguard-verifier/`, replacing the placeholder folder.
2. Verify no-internet loading:
   ```bash
   python3 -c "from transformers import AutoModelForSequenceClassification, AutoTokenizer; m=AutoModelForSequenceClassification.from_pretrained('models/clauseguard-verifier'); t=AutoTokenizer.from_pretrained('models/clauseguard-verifier'); print('loaded OK, no hub call')"
   ```
3. `git add models/clauseguard-verifier` (LFS-tracked via `*.safetensors`) and commit.